# Agentic Drone Inspection System -- Complete Technical Walkthrough

This notebook is a full architectural and operational guide to the Agentic Drone Inspection System built for solar farm monitoring. It covers every design decision, every data flow, every connection between components, and exactly what each piece of code does and why.

```
User command (natural language)
        |
        v
   FastAPI /run
        |
        v
  LangGraph Graph
        |
   supervisor -> planner -> executor -> safety
                                |
                         perception (per panel)
                                |
         reflection -> report -> supervisor (loop)
        |
        v
   JSON API response -> Browser UI
```

Read top to bottom. Each section builds on the previous one.

## 1. Problem Statement

A solar farm with an 8x8 grid of 64 zones needs regular inspection. Each zone has 5 individual solar panels -- 320 panels total. Defect categories:

| Category | Visual Signal |
|---|---|
| Clean | No defects, normal brightness |
| Bird-drop | Isolated bright patches |
| Dusty | Uniform grey film, reduced brightness |
| Electrical-Damage | Subtle texture variation, hot spots |
| Physical-Damage | Cracks, chips, high edge density |
| Snow-Covered | Uniform white coverage |

**Core design principle: panels are first-class citizens, not zones.** A zone is a physical container. Intelligence lives at the panel level: each panel has its own memory, graph position, feedback history, and risk score. Zones are only used for grouping in the report and UI -- they carry no verdict.

## 2. File Structure

```
drone_sim/
|
+-- main.py                         <- FastAPI app + UI server
+-- ui.html                         <- Mission control frontend
+-- .env                            <- GROQ_API_KEY, LANGCHAIN_API_KEY
|
+-- data/solar_farm/
|   +-- zone_0_0/                   <- 5 images per zone
|   |   +-- bird-drop_0000.jpg
|   |   +-- clean_0001.jpg
|   |   +-- dusty_0002.jpg
|   |   +-- bird-drop_0003.jpg
|   |   +-- clean_forced_0000.jpg
|   +-- zone_0_1/ ...
|   +-- zone_7_7/                   <- 64 zones x 5 images = 320 total
|
+-- docs/                           <- RAG knowledge base
|   +-- solar_inspection_guide.txt
|   +-- defect_types.txt
|
+-- chroma_db/                      <- Auto-created by ingest.py
+-- zone_memory.json                <- Zone-level episodic memory (macro)
+-- panel_memory.json               <- Panel-level episodic memory (primary)
+-- panel_feedback.json             <- Per-panel calibration feedback
+-- feedback_signals.json           <- Zone-level feedback (backward compat)
+-- trace_logs.json                 <- Local observability trace
|
+-- app/
    +-- core/llm.py                 <- Groq LLM init
    +-- agents/
    |   +-- supervisor.py           <- Orchestrator, mission lifecycle
    |   +-- planner.py              <- Zone assignment, panel risk scoring
    |   +-- executor.py             <- Drone sim, panel capture loop
    |   +-- perception.py           <- Hybrid perception pipeline
    |   +-- safety.py               <- Panel-level safety flags
    |   +-- reflection.py           <- Mission scoring, feedback storage
    |   +-- report.py               <- LLM-generated panel report
    +-- graph/
    |   +-- builder.py              <- LangGraph StateGraph definition
    |   +-- nodes.py                <- Agent wrappers for graph nodes
    |   +-- state.py                <- AgentState TypedDict
    +-- memory/
    |   +-- panel_memory.py         <- Per-panel history (PRIMARY)
    |   +-- panel_graph.py          <- Panel spatial graph (320 nodes)
    |   +-- panel_graph_rag.py      <- Panel GraphRAG queries
    |   +-- panel_feedback_store.py <- Per-panel calibration metrics
    |   +-- zone_memory.py          <- Zone summaries (macro layer)
    |   +-- zone_graph.py           <- 8x8 zone spatial graph
    |   +-- graph_rag.py            <- Zone-level GraphRAG
    |   +-- feedback_store.py       <- Zone-level feedback (compat)
    +-- rag/
    |   +-- ingest.py               <- Builds Chroma vector store
    |   +-- retriever.py            <- Feature-to-query RAG lookup
    +-- tools/
    |   +-- drone_tools.py          <- DroneEnv, swarm bus, battery
    +-- observability/
        +-- tracer.py               <- Local JSON trace writer
```

## 3. System Architecture

The system is a **stateful multi-agent loop** built on LangGraph. Every mission runs the same graph topology but adapts based on accumulated memory.

```
+---------------------------------------------------------------------+
|                     LANGGRAPH STATE MACHINE                         |
|                                                                     |
|  +-------------+                                                    |
|  |  SUPERVISOR |<-----------------------------------------+        |
|  | decides:    |                                           |        |
|  | plan/end/   |                                           |        |
|  | replan      |                                           |        |
|  +------+------+                                           |        |
|         | 'plan'                                           |        |
|         v                                                  |        |
|  +-------------+  reads: panel_memory                     |        |
|  |   PLANNER   |         panel_graph                      |        |
|  |             |         panel_feedback_store             |        |
|  |             |         zone_memory, zone_graph          |        |
|  +------+------+                                          |        |
|         | plan: List[Dict]                                |        |
|         v                                                 |        |
|  +-------------+  per panel:                             |        |
|  |  EXECUTOR   |  +----------------------------------+   |        |
|  |             |->|    PERCEPTION AGENT              |   |        |
|  |             |  |  System1: fast_classifier        |   |        |
|  |             |  |  System2: llm_classifier         |   |        |
|  |             |  |    reads: Chroma RAG             |   |        |
|  |             |  |           panel_graph_rag        |   |        |
|  |             |  |           panel_memory           |   |        |
|  |             |  |  Meta-Reasoner: final decision   |   |        |
|  |             |  +----------------------------------+   |        |
|  |             |  writes: panel_memory per panel         |        |
|  |             |          zone_memory after panel 4      |        |
|  |             |  swarm: zone_done, skip_zone msgs       |        |
|  +------+------+                                         |        |
|         | analysis: List[Dict] (panel-level)             |        |
|         v                                                |        |
|  +-------------+  flags panels                          |        |
|  |   SAFETY    |  approved / escalated / aborted        |        |
|  +------+------+  (always -> reflection)                |        |
|         v                                                |        |
|  +-------------+  score = high_conf/total               |        |
|  | REFLECTION  |  writes: panel_feedback_store          |        |
|  |             |  generates: DPO pairs                  |        |
|  +------+------+                                        |        |
|         v                                               |        |
|  +-------------+  LLM-generated panel report           |        |
|  |   REPORT    |  no zone verdicts, panels only        |        |
|  +------+------+                                        |        |
|         +------------------------------------------------+        |
+---------------------------------------------------------------------+
```

### What makes this agentic, not just a pipeline

1. **Supervisor loops** -- if mission_score < 0.5, it replans and sends drones out again
2. **Planner adapts** -- uncertain panels from feedback are auto-added and scored higher
3. **Drones communicate** -- zone_done triggers skip without central direction
4. **Battery failures** -- supervisor detects forced returns and logs for reassignment
5. **Perception uses context** -- same image, different result depending on panel history and neighbours

## 4. Shared State -- The Agent Contract

LangGraph routes a single `AgentState` dictionary through every node. Each agent reads from it and returns a partial update.

```python
class AgentState(TypedDict):
    user_input: str             # 'Inspect zone_0_0 zone_1_1'
    plan: List[Dict]            # planner output: task dicts
    execution_log: List[str]    # [MOVE], [PANEL], [DOCK] events
    drone_position: str
    images_captured: List[str]
    analysis: List[Dict]        # PANEL-LEVEL results, one per panel
    next_step: str
    safety_status: str          # 'approved' | 'escalated' | 'aborted'
    safety_flags: List[Dict]    # per-panel flag objects
    mission_score: float        # high_conf_panels / total_panels
    feedback_signal: List[Dict] # per-panel DPO pairs
    report: Optional[str]
    supervisor_decision: str    # 'plan' | 'end'
    supervisor_notes: str       # supervisor broadcasts to planner
    iteration: int
    swarm_messages: List[Dict]  # agent-to-agent msgs for UI
```

The `analysis` field is the spine of the system. Every downstream agent reads it:

```python
# One entry per panel -- zones have no entry
{
    'drone':       'drone_1',
    'zone':        'zone_0_0',   # grouping only
    'panel_index': 2,
    'panel_id':    'zone_0_0_p2',
    'image':       '/data/solar_farm/zone_0_0/clean_0001.jpg',
    'analysis': {
        'class':      'Clean',
        'confidence': 0.92,
        'reasoning':  'Dataset label: Clean. Features consistent...',
        'meta_note':  'filename + system agreed'
    }
}
```

## 5. LangGraph -- Node Connections

```python
# builder.py -- the graph topology

builder.set_entry_point('supervisor')

# Conditional: supervisor decides plan vs end
def route_supervisor(state):
    if state['supervisor_decision'] == 'end':
        return END      # must return the END constant, not string 'end'
    return 'planner'

builder.add_conditional_edges('supervisor', route_supervisor)

# Fixed edges
builder.add_edge('planner',    'executor')
builder.add_edge('executor',   'safety')

# Safety ALWAYS goes to reflection -- score must be computed even on abort
def route_safety(state):
    return 'reflection'   # unconditional

builder.add_conditional_edges('safety', route_safety, {'reflection': 'reflection'})

builder.add_edge('reflection', 'report')
builder.add_edge('report',     'supervisor')   # loop back
```

### Full traversal -- normal mission

```
supervisor  -> plan (iteration 1)
planner     -> builds 32-task plan for 4 zones
executor    -> runs plan, captures 20 panels, writes panel_memory
safety      -> 0 critical panels -> approved
reflection  -> 18/20 high-conf -> mission_score = 0.90
report      -> generates panel report
supervisor  -> score 0.90 >= 0.5 -> END
```

### Traversal -- low score triggers replan

```
supervisor  -> plan (iteration 1)
...mission runs...
reflection  -> 8/20 high-conf -> mission_score = 0.40
report      -> generates report
supervisor  -> score 0.40 < 0.5, iteration 1 < 3 -> replan
              clears report, sets supervisor_notes
planner     -> re-reads feedback, adds uncertain zones, re-scores
...second mission...
reflection  -> 15/20 high-conf -> mission_score = 0.75
supervisor  -> score 0.75 >= 0.5 -> END
```

## 6. Supervisor -- Orchestrator

The supervisor is the entry point and loop controller. It is the only agent that decides whether the mission continues or ends.

```
On every call, supervisor reads: iteration, mission_score, report, execution_log

if report is None AND iteration == 0:
    -> First call. Start mission.
    -> supervisor_decision = 'plan'
    -> supervisor_notes = 'Initial mission. Drones at dock with full battery.'

if report is not None:
    -> A full cycle completed.
    -> if mission_score < 0.5 AND iteration < 3:
           -> Replan. Clear report. Set notes explaining why.
       else:
           -> END
```

### Battery-triggered reassignment detection

```python
# supervisor reads execution_log looking for forced returns
battery_critical = [line for line in execution_log if '[BATTERY CRITICAL]' in line]

if battery_critical:
    supervisor_notes = (
        f'Battery critical on iteration {iteration}. '
        f'Supervisor reassigning incomplete zones to available drone.'
    )
    supervisor_decision = 'plan'   # replan to cover unfinished zones
```

The supervisor does not micromanage assignments -- that is the planner's job. The supervisor only decides: **keep going or stop**. `supervisor_notes` is written into shared state; the planner reads it, logs it, and the UI surfaces it as a blue block at the top of the Swarm tab.

### What supervisor communicates and to whom

```
supervisor -> planner:   via state['supervisor_notes'] + state['supervisor_decision']
supervisor -> UI:        via state['supervisor_notes'] surfaced in API response
supervisor -> graph:     via return {'supervisor_decision': 'end'} -> END
```

## 7. Planner -- Panel-Aware Zone Assignment

The planner converts a zone list into a concrete execution plan. It reads five memory sources to score and prioritise.

### Step 1 -- Add high-uncertainty zones automatically

```python
uncertain_panels = get_high_uncertainty_panels(threshold=0.35)
# Returns panels where fast/slow disagreed >35% of past missions
# e.g. [{panel_id: 'zone_2_3_p2', disagreement_score: 0.6, ...}]

for up in uncertain_panels:
    zone = up['panel_id'].rsplit('_p', 1)[0]   # 'zone_2_3'
    if zone not in zones:
        zones.append(zone)   # auto-add without user instruction
```

This is the self-improvement loop. After a mission where the system was uncertain about some panels, those panels' zones are automatically re-inspected next mission.

### Step 2 -- Panel-level risk scoring

```python
for zone in zones:
    zone_risk = 0.0
    for p_idx in range(5):
        panel_risk = 0.0

        # Source 1: panel's own history (panel_memory.json)
        history = get_panel(zone, p_idx)
        if history:
            last = history[-1]
            if last['class'] in ('Physical-Damage', 'Electrical-Damage'):
                panel_risk += last['confidence'] * 1.5
            elif last['class'] in ('Bird-drop', 'Dusty'):
                panel_risk += last['confidence'] * 0.8

        # Source 2: disagreement feedback (panel_feedback.json)
        feedback = get_panel_feedback(zone, p_idx)
        if feedback:
            panel_risk += feedback['disagreement_score'] * 0.4
            panel_risk += feedback['avg_calibration_error'] * 0.3

        # Source 3: adjacent panel history (cross-zone aware)
        for neighbour in get_panel_neighbors(zone, p_idx):
            n_last = get_panel(neighbour['zone'], neighbour['panel_index'])
            if n_last and n_last[-1]['class'] in ('Physical-Damage', 'Electrical-Damage'):
                panel_risk += n_last[-1]['confidence'] * 0.6

        zone_risk += panel_risk

zone_risk_scores.sort(key=lambda x: x[1], reverse=True)
# Highest-risk zones assigned to drone_1 (primary)
```

### Step 3 -- Build the execution plan

```python
for i, (zone, risk) in enumerate(zone_risk_scores):
    primary   = drones[i % 2]        # alternates: drone_1, drone_2, ...
    secondary = drones[(i+1) % 2]

    # Primary: full 5-panel inspection, always P0->P1->P2->P3->P4
    plan.append({'drone': primary,   'action': 'move_to',       'zone': zone})
    for p in range(5):
        plan.append({'drone': primary, 'action': 'capture_panel', 'zone': zone, 'panel_index': p})
    plan.append({'drone': primary,   'action': 'return_to_dock', 'zone': zone})

    # Secondary: same zone -> executor will skip via swarm
    # This generates real zone_done/skip_zone messages
    plan.append({'drone': secondary, 'action': 'move_to',       'zone': zone})
    plan.append({'drone': secondary, 'action': 'capture_panel', 'zone': zone, 'panel_index': 0})
    plan.append({'drone': secondary, 'action': 'return_to_dock', 'zone': zone})
```

Panels scan in natural P0->P4 order. Within-zone risk reordering was removed because it scrambled the log and animation. Zone priority (which zone first) is still risk-driven.

## 8. Executor -- Running the Plan

The executor iterates the plan with a `while i < len(plan)` loop (not for-each) so it can fast-forward `i` when skipping zones.

### Per-task flow

```
for each task:
  1. Drain swarm inbox for this drone
  2. Check if zone in completed_zones AND this drone hasn't done it:
       -> find who completed it from swarm messages
       -> log [SWARM] skip
       -> post skip_zone back
       -> while plan[i].zone == zone AND plan[i].drone == drone_id: i++
       -> continue (skip to next task)
  3. Execute: move_to | capture_panel | return_to_dock
```

### Battery model

```
BATTERY_PER_MOVE    = 8%  per zone traversal (in DroneEnv.move_to)
BATTERY_PER_CAPTURE = 1%  per panel (in executor capture_panel)
BATTERY_LOW_THRESH  = 25% -> post battery_low to partner
BATTERY_CRIT_THRESH = 15% -> forced return to dock

Typical 4-zone mission:
  4 moves * 8% = 32%
  20 captures * 1% = 20%
  Total: ~52% used, drone finishes with ~48%
  -> Battery critical (15%) not normally hit in 4-zone missions
```

### Capture panel -- the full sequence

```python
elif action == 'capture_panel':
    # Emit scanning header on first panel of this zone
    scan_key = f'{drone_id}_{zone}'
    if scan_key not in zone_scan_started:
        logs.append(f'[SCAN_START] {drone_id} zone={zone}')
        zone_scan_started.add(scan_key)

    image_path = get_panel_image(zone, panel_index)
    # files[panel_index % len(files)] -- deterministic mapping

    analysis = perception_agent(image_path, zone=zone, panel_index=panel_index)

    if analysis['confidence'] < 0.55:
        retry = perception_agent(image_path, zone=zone, panel_index=panel_index)
        if retry['confidence'] > analysis['confidence']:
            analysis = retry

    logs.append(f'[PANEL] {drone_id} zone={zone} panel={panel_index} ...')

    # Write to panel_memory IMMEDIATELY
    update_panel(zone, panel_index, panel_record)
    refresh_panel_graph()   # <- panel N+1 can now see panel N's result

    # After panel 4: zone complete
    if panel_index == 4:
        update_zone(zone, zone_summary)      # zone macro layer
        drone.complete_zone(zone)            # posts zone_done to partner
        completed_zones.add(zone)
        logs.append(f'[ZONE_DONE] {drone_id} zone={zone}')
```

The `refresh_panel_graph()` call after each panel is critical. Panel 0's result is immediately visible to panel 1's GraphRAG query as a neighbour. By the time panel 4 is analyzed, it has context from panels 0-3 already in the graph.

## 9. Perception Agent -- Cognitive Hybrid Architecture

The perception agent runs three systems in sequence and applies a meta-reasoner to produce the final classification.

### Input

```python
perception_agent(
    image_path  = '/data/solar_farm/zone_0_0/bird-drop_0036.jpg',
    zone        = 'zone_0_0',
    panel_index = 1
)
```

### Step 1 -- OpenCV Feature Extraction (11 features)

```
brightness       = mean(gray)
red/blue/green   = channel means
edge_density     = sum(canny_edges) / total_pixels
texture_variance = var(gray)
local_roughness  = mean(abs(pixel - local_mean))  [5x5 kernel]
white_ratio      = pixels > 200 / total
dark_ratio       = pixels < 50  / total
red_ratio        = mean_r / (mean_r + mean_g + mean_b)
blue_ratio       = mean_b / (mean_r + mean_g + mean_b)
```

### Step 2 -- Filename Prior (simulation stand-in for VLM)

```python
# Split filename on both '-' and '_', match keyword sets
# bird-drop_0036.jpg -> tokens: {'bird', 'drop', '0036'} -> Bird-drop
# clean_forced_0000.jpg -> tokens: {'clean', 'forced', '0000'} -> Clean

KEYWORD_MAP = [
    ({'physical', 'damage'}, 'Physical-Damage'),
    ({'electrical', 'damage'}, 'Electrical-Damage'),
    ({'snow', 'covered'}, 'Snow-Covered'),
    ({'bird', 'drop'}, 'Bird-drop'),
    ({'clean'}, 'Clean'),
    ({'dusty'}, 'Dusty'),
]
# Returns: ('Bird-drop', 0.92)
```

In production: replace `class_from_filename()` with a VLM API call. Nothing else changes.

### Step 3 -- System 1: Fast Classifier (rule-based)

```python
if white   > 0.45:                 return {'class': 'Snow-Covered',      'confidence': 0.78}
if tex_var > 4500 and edge < 70:   return {'class': 'Electrical-Damage', 'confidence': 0.62}
if edge    > 75   and tex > 2000:  return {'class': 'Physical-Damage',   'confidence': 0.60}
if 0.08 < white < 0.4 and dark < 0.15 and bright > 110:
                                   return {'class': 'Bird-drop',         'confidence': 0.56}
if bright  < 110 and tex > 2500 and edge < 70:
                                   return {'class': 'Dusty',             'confidence': 0.58}
return                                    {'class': 'Clean',             'confidence': 0.52}
```

Fast, deterministic, no API call. Acts as System 1 in the dual-process model.

### Step 4 -- System 2: LLM Classifier (Groq, full context)

The LLM receives a prompt built from four sources:

```
=== IMAGE FEATURES ===
{ brightness: 113.4, edge_density: 81.2, ... }

Dataset label (ground truth): Bird-drop (file: bird-drop_0036.jpg)

=== THIS PANEL'S HISTORY ===      <- panel_memory.json
  - Bird-drop (conf 0.77, meta: filename override)

=== ADJACENT PANEL CONTEXT ===    <- panel_graph_rag
  - Adjacent panel zone_0_0_p0 (within zone) history: Bird-drop (0.77)
  - Neighbour zone_0_1_p4 (cross zone horizontal) history: Dusty

=== DOMAIN KNOWLEDGE ===          <- Chroma RAG
  - Bird droppings create isolated bright patches, white_ratio 0.05-0.35
  - High edge density suggests cracks or physical breaks
```

### Step 5 -- Meta-Reasoner (cognitive decision layer)

Six signals, in priority order:

```
1. Filename label (0.92 confidence)
   IF label present AND (fast OR slow agrees):
       -> use filename class, boost confidence
       -> meta_note: 'filename + system agreed'
   IF label present AND neither system agrees:
       -> filename still wins (it's ground truth)
       -> meta_note: 'filename override'

2. Both systems agree on same class:
   -> take higher confidence
   -> meta_note: 'both systems agreed'

3. Panel GraphRAG supports slow but not fast:
   -> slow wins
   -> meta_note: 'slow + panel_graph agree'

4. Fast historically unreliable on THIS panel:
   (overridden 2+ times in panel_memory history)
   -> slow wins regardless of confidence
   -> meta_note: 'fast historically unreliable on this panel'

5. Damage corridor detected:
   (2+ adjacent panels have damage history)
   IF slow predicts damage: slow wins
   -> meta_note: 'damage corridor -- deferred to slow'

6. Confidence delta > 0.2:
   -> higher confidence wins
   IF slow predicts risky class (Physical/Electrical/Bird-drop): slow wins
   Final fallback: higher confidence wins
   -> meta_note: 'confidence tiebreak'
```

This is cognitive because it asks: *given everything known about this panel -- its history, neighbours, domain knowledge, and classifier reliability -- which system to trust?* The answer can differ from pure confidence comparison.

## 10. Memory Architecture -- Two Parallel Layers

```
PANEL-LEVEL (primary intelligence)

panel_memory.json                    panel_feedback.json
  zone_0_0_p0: [                       zone_0_0_p0_meta: {
    {class: Bird-drop, conf: 0.77},       disagreement_score: 0.4,
    {class: Bird-drop, conf: 0.88}        avg_confidence: 0.82,
  ]                                       class_history: {Bird-drop: 2}
  zone_0_0_p1: [...]                    }

  written by: executor (after each capture)
  written by: reflection (feedback signals)
  read by:    perception (panel history for LLM prompt)
  read by:    planner (risk scoring per panel)

panel_graph (NetworkX, in-process)
  320 nodes: zone_0_0_p0 ... zone_7_7_p4
  Within-zone edges:        p0-p1-p2-p3-p4 (4 per zone x 64 = 256)
  Cross-zone horizontal:    p4(zone_i_j) <-> p0(zone_i_j+1)
  Cross-zone vertical:      p0..p4(zone_i_j) <-> p0..p4(zone_i+1_j)

  panel_graph_rag overlays defect nodes:
    zone_0_0_p0 --[weight=0.88]--> defect_Bird-drop

  refreshed: after each panel write (executor calls refresh_panel_graph())
  read by:   perception (neighbour context for LLM prompt)
  read by:   planner (adjacent panel risk propagation)

ZONE-LEVEL (macro layer)

zone_memory.json                     feedback_signals.json
  zone_0_0: [{                         zone_0_0_meta: {
    class: Bird-drop,                    disagreement_score: 0.3
    panel_breakdown: {Bird:3, Clean:2}   }
  }]

  written by: executor (after panel 4 of each zone)
  read by:    planner (zone-level risk from neighbours)
  read by:    zone_graph_rag (zone GraphRAG)
```

### Why both layers exist

Zone memory is fast to read and gives a coarse signal for zone prioritisation. Panel memory gives fine-grained per-panel history that the LLM receives as explicit context. A zone where all 5 panels are consistently clean generates low zone_risk. A zone where 2 panels oscillate between Physical-Damage and Clean generates high panel_risk for those specific panels, and the planner prioritises them.

### panel_graph -- the physical topology

```
zone_0_0:  [p0][p1][p2][p3][p4]     zone_0_1: [p0][p1][p2][p3][p4]

Within zone_0_0: p0-p1, p1-p2, p2-p3, p3-p4
Cross-zone horiz: zone_0_0_p4 <-> zone_0_1_p0
Cross-zone vert:  zone_0_0_p0 <-> zone_1_0_p0  (and p1..p4)

Result: zone_0_0_p4 sees zone_0_1_p0 as a cross-zone neighbour.
If zone_0_1_p0 has Physical-Damage history, zone_0_0_p4's GraphRAG
query surfaces this as: 'Adjacent panel zone_0_1_p0 (cross zone
horizontal) has history of Physical-Damage (confidence 0.85)'
```

## 11. Swarm Communication -- Peer-to-Peer Drone Messaging

Drones communicate through a shared in-process message bus in `drone_tools.py`.

```python
_swarm_inbox = {'drone_1': [], 'drone_2': []}

def post_swarm_message(from_drone, to_drone, msg_type, zone, message):
    _swarm_inbox[to_drone].append({
        'from': from_drone, 'type': msg_type,
        'zone': zone, 'message': message
    })

def read_swarm_messages(drone_id):
    msgs = _swarm_inbox.get(drone_id, [])
    _swarm_inbox[drone_id] = []   # drain -- consumed once
    return msgs
```

### Message types

| Type | Sender | Receiver | Trigger | Effect |
|---|---|---|---|---|
| zone_done | primary | secondary | After panel 4 | Secondary skips zone |
| skip_zone | secondary | primary | After zone_done | Acknowledgement |
| battery_low | any | partner | Battery < 25% | Partner logs warning |

### The zone_done flow in detail

```
drone_1 captures panel 4 of zone_0_0
  -> executor: drone.complete_zone('zone_0_0')
       -> DroneEnv.complete_zone():
            self.zones_done.add('zone_0_0')
            post_swarm_message('drone_1', 'drone_2', 'zone_done', 'zone_0_0', ...)
  -> completed_zones.add('zone_0_0')

executor reaches drone_2's move_to zone_0_0 task:
  -> read_swarm_messages('drone_2')
  -> no zone_done yet (messages drain once read)
  -> zone_0_0 IS in completed_zones
  -> find completer from all_swarm_msgs: drone_1
  -> logs: '[SWARM] drone_2 skipping zone_0_0 -- done by drone_1'
  -> post_swarm_message('drone_2', 'drone_1', 'skip_zone', 'zone_0_0', ...)
  -> fast-forward i past all drone_2/zone_0_0 tasks
  -> continue to drone_2's next zone
```

### Why this is genuine peer-to-peer

The secondary drone is not told by any central controller to skip. It receives a message from the primary, checks against its own knowledge (completed_zones set), makes its own skip decision, and posts an acknowledgement back. The executor loop handles this generically -- it does not special-case any specific drone or zone combination.

The planner deliberately assigns both drones to every zone to generate this communication. Without the secondary assignment, the bus would be empty.

## 12. Safety, Reflection, and Report

### Safety Agent

Reviews every panel in `state['analysis']`. Operates at panel level -- no zone verdicts.

```
For each panel result:
  if confidence < 0.55:
      flag: 'low_confidence'
      low_conf_count++
  if class in (Physical-Damage, Electrical-Damage) AND confidence > 0.7:
      flag: 'critical_damage_detected'
      critical_count++

Decisions (in order):
  critical_count >= 8  -> aborted   (extreme damage across farm)
  critical_count >= 3  -> escalated (human review needed)
  low_conf_count >= 3  -> escalated
  else                 -> approved

NOTE: safety ALWAYS routes to reflection.
      Even on abort, reflection runs so mission_score != 0.
```

### Reflection Agent

```python
for panel in analysis:
    total += 1
    if confidence > 0.7: high_conf += 1

    # disagreement = fast and slow did not cleanly agree
    disagreement = meta_note not in ('both systems agreed',
                                      'filename + system agreed', '')

    signal = {
        'zone': zone, 'panel_index': panel_index,
        'class': defect_class, 'confidence': confidence,
        'disagreement': disagreement,
        'dpo_preferred': reasoning,   # for future fine-tuning
        'dpo_context': f'panel={zone}_p{panel_index}, ...'
    }
    store_panel_feedback(zone, panel_index, signal)
    # -> updates disagreement_score, avg_confidence, class_history
    # -> planner reads this next mission to boost uncertain zones

mission_score = high_conf / total
```

### Report Agent

Groups panels by zone for readability only. The LLM prompt explicitly says:
*'Do NOT assign a single verdict to the zone. Each panel stands alone.'*

Report sections:
1. Executive Summary
2. Panel-Level Findings (each panel P0-P4 per zone individually)
3. Critical Panels (conf > 0.7 and damage class)
4. Damage Patterns (adjacent panels with same defect)
5. Safety Flags and HITL Recommendations
6. Recommended Actions per panel

## 13. RAG System -- Domain Knowledge Retrieval

### SolarEmbeddingFunction -- custom 32-dim TF-IDF encoder

```python
VOCAB = ['dust', 'dusty', 'snow', 'white', 'crack', 'edge', 'damage',
         'electrical', 'texture', 'hotspot', 'clean', 'defect', ...]

IDF = {'crack': 2.0, 'electrical': 2.0, 'hotspot': 2.0, ...}  # rare terms boosted

def _encode(text):
    tokens = tokenize(text)
    vec = [tf(term, tokens) * idf(term) for term in VOCAB]
    return l2_normalize(vec)  # cosine similarity works correctly
```

No sentence-transformers, no BERT, no model download. A 32-dim domain-specific vector captures the same discriminative signal as a 384-dim general encoder for this specific vocabulary.

### Ingest (one-time)

```bash
python -m app.rag.ingest
```

Reads docs/*.txt -> 500-char chunks with 50-char overlap -> SolarEmbeddingFunction -> Chroma SQLite + vector index at chroma_db/

### Retrieval (every perception call)

```python
def retrieve_context(features, k=3):
    query_parts = []
    if features['white_ratio'] > 0.5:
        query_parts.append('snow coverage white panel')
    if features['edge_density'] > 0.2:
        query_parts.append('physical damage cracks edge detection')
    if features['brightness'] < 100:
        query_parts.append('dust accumulation low brightness')

    docs = db.similarity_search(' '.join(query_parts), k=3)
    return [doc.page_content for doc in docs]
```

Retrieved chunks land in the LLM prompt as Domain Knowledge, grounding the classification in inspection principles rather than pure feature pattern matching.

## 14. Mission Control UI

The UI is a single HTML file served by FastAPI. Computation happens on the server; the UI replays the result as a stepped animation.

### Flow

```
User types command -> POST /run
Server: graph.invoke() -> JSON response with execution_log, analysis, swarm_messages
Browser: animates execution_log line by line with sleep() delays
  -> each [PANEL] event colors that specific panel cell on the grid
  -> Analysis tab populated with P0-P4 rows per zone
  -> Swarm tab shows zone_done/skip_zone/battery messages
  -> Report tab shows LLM text
```

### Log event format

| Event | Format | UI action |
|---|---|---|
| [MOVE] | [MOVE] drone_1 -> zone_0_0 [battery:92%] | Place dot, start scanning anim, update battery |
| [SCAN_START] | [SCAN_START] drone_1 zone=zone_0_0 | Emit 'scanning zone panels:' header |
| [PANEL] | [PANEL] drone_1 zone=zone_0_0 panel=2 class=Clean confidence=0.92 | Color P2 of zone_0_0 |
| [ZONE_DONE] | [ZONE_DONE] drone_1 zone=zone_0_0 | Remove scanning anim, emit checkmark |
| [DOCK] | [DOCK] drone_1 returned [recharged:100%] | Remove dot, reset battery to 100% |
| [SWARM] | [SWARM] drone_2 skipping zone_0_0 | Log swarm event |
| [BATTERY CRITICAL] | [BATTERY CRITICAL] drone_1 battery 14% | Red log, battery bar drops |

### Panel coloring

Each of the 5 panels in a zone is an independent DOM element `panel-zone_0_0-2`. When a [PANEL] event arrives, `setPanelClass(zone, panelIndex, className)` applies a CSS class:

```css
.panel.clean      { background: #0f2d1a; border-color: #238636; }
.panel.bird-drop  { background: #2a1e00; border-color: #9e6a03; }
.panel.dusty      { background: #2d2200; border-color: #bb8009; }
.panel.electrical { background: #1d0f2d; border-color: #8957e5; }
.panel.physical   { background: #2d0f0f; border-color: #da3633; }
.panel.snow       { background: #0f1d2d; border-color: #388bfd; }
```

Panels within a zone can show different colors -- zone_0_0 may have p0=Bird-drop, p2=Clean, p4=Dusty, each in its own color.

### Analysis tab structure

```
zone_0_0                          drone_1 * 5 panels
  [P0] Bird-drop  --------  88%  label+sys   [img]
  [P1] Bird-drop  --------  88%  label+sys   [img]
  [P2] Clean      ---------  92%  both        [img]
  [P3] Clean      ---------  95%  label+sys   [img]
  [P4] Dusty      --------  85%  override    [img]

Zone header has NO class badge. Zones carry no verdict.
P0..P4 badge is colored by the panel's own class.
```

## 15. Observability -- Two-Layer Tracing

### Layer 1 -- Local JSON trace

```python
# tracer.py -- called from every agent
def log_event(agent, step, data):
    trace = {
        'timestamp': time.time(),
        'agent': agent,
        'step': step,
        'data': data
    }
    logs.append(trace)
    json.dump(logs, file)

# Example events:
log_event('perception', 'features', {brightness: 113.4, ...})
log_event('perception', 'final_decision', {fast: ..., slow: ..., final: ...})
log_event('planner', 'zone_priority', [('zone_5_5', 2.1), ('zone_0_0', 1.4)])
log_event('safety', 'critical_panels', 2)
```

Available offline without any API key. Full execution history in one JSON file.

### Layer 2 -- LangSmith cloud tracing

```python
from langsmith import traceable

@traceable(name='perception_agent', run_type='chain')
def perception_agent(image_path, zone, panel_index): ...

@traceable(name='llm_classifier', run_type='llm')
def llm_classifier(features, rag_context, ...): ...

@traceable(name='rag_retrieve_context', run_type='retriever')
def retrieve_context(features, k=3): ...

@traceable(name='panel_graph_rag_query', run_type='retriever')
def query_panel_graph(zone, panel_index): ...
```

With `LANGCHAIN_TRACING_V2=true` in .env, LangGraph auto-traces the graph. @traceable adds function-level spans.

### Trace tree in LangSmith

```
graph.invoke (root)
  supervisor_agent
  planner_agent
  executor_agent
    perception_agent (x20 panels)
      extract_features       <- OpenCV timing
      rag_retrieve_context   <- Chroma query + returned chunks
      panel_graph_rag_query  <- Graph traversal results
      fast_classifier        <- Rule result
      llm_classifier         <- Full prompt + Groq response + tokens
      meta_reasoner          <- Decision path taken
  safety_agent
  reflection_agent
  report_agent
```

### .env

```
GROQ_API_KEY=gsk_xxxx
LANGCHAIN_API_KEY=lsv2_pt_xxxx   <- from smith.langchain.com/settings
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=agentic-drone-inspector
```

## 16. Complete Data Flow -- One Mission End to End

```
USER: 'Inspect zone_0_0 zone_1_1 zone_3_4 zone_5_5'

POST /run -> graph.invoke(initial_state)

SUPERVISOR (iteration 0)
  report=None, iteration=0
  -> supervisor_decision='plan', iteration=1
  -> notes: 'Initial mission. Drones at dock.'

PLANNER
  reads: panel_memory -> no history yet (first mission)
  reads: panel_feedback -> no uncertainty yet
  computes: zone_risk for all 4 zones (all 0.0 on first mission)
  builds: 32-task plan
  logs: zone assignment per drone

EXECUTOR -- zone_5_5 (drone_1)
  move_to zone_5_5 [battery: 92%]
  capture panel 0:
    image: bird-drop_0181.jpg
    filename_class: Bird-drop (0.92)
    RAG: 'Bird droppings create isolated bright patches...'
    panel_graph: no history yet (first panel of mission)
    fast: Physical-Damage (0.60) [edge=60, tex=4100]
    slow: Bird-drop (0.88) [label present, features consistent]
    meta: filename + slow agree -> Bird-drop (0.88)
    -> update_panel('zone_5_5', 0, result)
    -> refresh_panel_graph()   <- p0 now visible to p1's query
  capture panel 1:
    panel_graph: 'Adjacent p0 (within zone) has history of Bird-drop (0.88)'
    -> LLM sees this neighbour context
    -> Bird-drop (0.88)
  panels 2,3,4... (each sees growing neighbour context)
  panel 4:
    -> update_zone('zone_5_5', summary)
    -> drone.complete_zone() -> post zone_done to drone_2
    -> completed_zones.add('zone_5_5')
    -> [ZONE_DONE] drone_1 zone=zone_5_5
  return_to_dock [recharged: 100%]

EXECUTOR -- zone_5_5 (drone_2)
  move_to zone_5_5:
    -> zone_5_5 in completed_zones, drone_2 hasn't done it
    -> find completer: drone_1
    -> [SWARM] drone_2 skipping zone_5_5 -- done by drone_1
    -> post skip_zone to drone_1
    -> fast-forward past all drone_2/zone_5_5 tasks

(repeat for zone_1_1, zone_0_0, zone_3_4 with alternating drone)

EXECUTOR returns:
  analysis: 20 panel dicts
  execution_log: ~80 lines ([MOVE],[SCAN_START],[PANEL]x5,[ZONE_DONE],[DOCK] per zone)
  swarm_messages: 4x zone_done + 4x skip_zone + 4x duplicate_skip

SAFETY
  20 panels reviewed
  e.g. 1 Physical-Damage panel at 0.82 -> critical_count=1
  1 < 3 -> approved
  flags: [{'panel': 'zone_5_5_p4', 'reason': 'critical_damage_detected', ...}]

REFLECTION
  18 of 20 panels confidence > 0.7
  mission_score = 18/20 = 0.90
  stores 20 panel feedback signals to panel_feedback.json
  (any panels where meta_note not in 'both agreed/filename+sys/empty'
   get disagreement=True, boosting next-mission risk score)

REPORT
  groups 20 panels by zone
  LLM generates: Executive Summary + per-panel findings + critical list
  'zone_5_5 panel P4: Physical-Damage, confidence 82%, recommend repair.'

SUPERVISOR (call 2)
  report not None, mission_score=0.90
  0.90 >= 0.5 -> END

graph.invoke returns final state
main.py extracts and returns JSON:
  {report, mission_score:0.90, safety_status:'approved',
   analysis:[20 panels], execution_log:[~80 lines],
   swarm_messages:[12 msgs], supervisor_notes:'...'}

UI animates -> 8x8 grid panels light up -> Analysis tab -> Report tab
```

## 17. Connections Table -- Who Reads What From Whom

### Writes

| Component | Writes To | What |
|---|---|---|
| executor | panel_memory.json | Per-panel result after each capture |
| executor | zone_memory.json | Zone summary after panel 4 |
| executor | swarm_inbox | zone_done, skip_zone, battery_low |
| reflection | panel_feedback.json | Disagreement score, DPO pair |
| reflection | feedback_signals.json | Zone feedback (backward compat) |
| ingest.py | chroma_db/ | Encoded inspection document chunks |
| tracer.py | trace_logs.json | Every log_event() call |

### Reads

| Component | Reads From | Why |
|---|---|---|
| planner | panel_memory.json | Last finding per panel -> risk score |
| planner | panel_feedback.json | Disagreement score -> boost uncertain zones |
| planner | panel_graph.py | Neighbour panel histories -> cross-zone risk |
| planner | zone_memory.json | Zone summaries -> macro risk signal |
| planner | zone_graph.py | Zone neighbours -> zone-level risk prop |
| perception | panel_memory.json | Panel's past findings -> LLM history section |
| perception | panel_graph_rag.py | Neighbour defect history -> LLM context |
| perception | chroma_db/ | Domain knowledge chunks -> LLM RAG section |
| safety | state[analysis] | All panel results -> flags and counts |
| reflection | state[analysis] | All panel results -> score and feedback |
| report | state[analysis] | All panel results -> LLM prompt input |
| supervisor | state[mission_score] | Decide to end or replan |
| supervisor | state[execution_log] | Detect battery critical for reassignment |

## 18. Pitfalls and Fixes

### LangGraph END constant

**Problem:** `KeyError: '__end__'` when routing to terminal state.
**Root cause:** Newer LangGraph requires returning the `END` constant from routing functions. Returning the string `'end'` with a path map fails because LangGraph internally uses `'__end__'`.
**Fix:** Remove the path map. Return `END` directly:

```python
def route_supervisor(state):
    if state['supervisor_decision'] == 'end': return END
    return 'planner'
builder.add_conditional_edges('supervisor', route_supervisor)
```

### mission_score 0% on abort

**Problem:** Aborted missions always showed Score: 0%.
**Root cause:** Safety routed directly to report on abort, skipping reflection. mission_score stayed at initial 0.0.
**Fix:** Safety always routes to reflection. `route_safety` returns `'reflection'` unconditionally.

### Duplicate log lines in UI

**Problem:** Each move/dock event appeared twice.
**Root cause:** Animate function had both a strict `[MOVE]` regex AND a loose fallback regex. Both matched the same line.
**Fix:** Add `^` anchors to all regexes. Remove all fallbacks. Unmatched lines silently skipped.

### Panel scan order scrambled

**Problem:** Log showed P3->P4->'scanning header'->P0->P1->P2.
**Root cause:** Planner reordered panels within a zone by risk score. The [SCAN_START] header fired on panel_index==0, which was no longer the first task.
**Fix:** Remove within-zone panel reordering. Panels always scan P0->P4. Zone priority stays risk-driven; within-zone order is natural.

### NumPy 2.0 breaking ChromaDB

**Problem:** `AttributeError: np.float_ was removed in NumPy 2.0`
**Fix:** Upgrade ChromaDB to >= 0.5.3 which supports NumPy 2.0.

### sentence-transformers causing RAM issues

**Problem:** Loading all-MiniLM-L6-v2 required ~500MB RAM, slow on laptop.
**Fix:** Custom SolarEmbeddingFunction -- 32-dim TF-IDF, zero model downloads, implements Chroma's EmbeddingFunction interface directly.

### All panels same class

**Problem:** All 5 panels in a zone classified identically.
**Root cause:** Fast classifier thresholds calibrated for normalized edge density (0-1) but dataset had raw values (40-90). Everything hit the Dusty branch.
**Fix:** Recalibrate thresholds to real dataset distributions. Add filename prior as ground truth anchor.

### Secondary drone scanning before skip

**Problem:** Secondary drone captured panel 0 before deciding to skip.
**Root cause:** Skip check only ran at start of each task, but the zone was added to completed_zones by the primary's panel-4 capture. Secondary's move_to ran before that happened.
**Fix:** Check `completed_zones` set at every task. When skip triggered, fast-forward `i` past all remaining tasks for that drone/zone combo using a while loop.

## 19. Techniques and Methods

**Multi-Agent Systems (LangGraph)**
Stateful directed graph where each node is an agent function. State passes through nodes, each partially updates it. Conditional edges implement routing. Compiled graph has built-in checkpointing and LangSmith observability.

**Cognitive Hybrid Architecture (Dual-Process Reasoning)**
Borrowed from cognitive science (Kahneman System 1/2). Fast classifier is rule-based, cheap, deterministic. LLM classifier is deliberate, expensive, context-aware. Meta-reasoner applies domain logic to decide which to trust -- not a simple confidence comparison but a multi-signal cognitive decision.

**Retrieval-Augmented Generation (RAG)**
LLM classifier grounded in external knowledge. Feature signals converted to natural language query; Chroma retrieves top-k document chunks; these appear in the LLM prompt as explicit context. Prevents reliance on LLM parametric knowledge alone.

**GraphRAG (Graph-Enhanced Retrieval)**
Two graph layers: Zone GraphRAG (64 zone nodes + defect nodes + spatial edges) and Panel GraphRAG (320 panel nodes + cross-zone edges). Queries surface what neighbouring panels have experienced historically, including damage corridor detection when 2+ adjacent panels show damage.

**Episodic Memory**
Each panel has a history list in panel_memory.json. The LLM receives the last 3 entries as explicit history in its prompt. The system remembers specific past events for specific entities (individual panels), not just aggregate statistics.

**Feedback Loop / Self-Improvement**
After each mission, reflection stores per-panel disagreement signals. Planner reads these next mission and boosts risk scores of uncertain panels. Over multiple missions: consistently correct panels generate no signal; volatile panels get more attention and richer LLM context.

**DPO Training Signal Generation**
Each reflection signal includes dpo_preferred (final reasoning text) and dpo_context (panel ID + class + confidence + meta_note). Formatted as (preferred, context) pairs for future Direct Preference Optimisation fine-tuning of the LLM classifier.

**Swarm Coordination**
Shared in-process message bus enables peer-to-peer communication. Drones post to each other's inboxes and drain them at each task. Zone_done triggers skip decisions; battery_low enables partner awareness. No central coordinator for the skip decision.

**TF-IDF Domain Embeddings**
Custom 32-dim TF-IDF embedding tuned to solar inspection vocabulary. IDF weights boost rare/important terms. L2 normalisation enables cosine similarity. No general-purpose model needed.

**LangSmith Observability**
@traceable decorators on all agents and key sub-functions. LangGraph auto-traces graph topology. Combined view: full execution tree with latency, inputs, outputs, token usage per LLM call.

## 20. How to Run

### Install

```bash
pip install fastapi uvicorn langchain langgraph langchain-groq langchain-chroma
pip install langchain-community langchain-text-splitters
pip install chromadb networkx opencv-python-headless numpy pillow
pip install langsmith python-dotenv
```

### .env

```
GROQ_API_KEY=gsk_xxxx
LANGCHAIN_API_KEY=lsv2_pt_xxxx
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=agentic-drone-inspector
```

### Start sequence

```bash
# Optional: clean slate
rm -rf chroma_db
rm -f zone_memory.json panel_memory.json feedback_signals.json panel_feedback.json trace_logs.json

# Build RAG index (one-time per docs/ change)
python -m app.rag.ingest

# Start server
uvicorn main:app --reload

# Open browser
# http://localhost:8000
```

### API endpoints

```
POST /run                             <- main mission endpoint
GET  /panel/{zone}/{panel_index}      <- panel memory history
GET  /panel-feedback/{zone}/{panel}   <- panel feedback metrics
GET  /panel-graph/{zone}/{panel}      <- panel GraphRAG insights
GET  /uncertain-panels                <- panels with disagreement > 0.3
GET  /memory/{zone}                   <- zone memory
GET  /feedback/{zone}                 <- zone feedback
GET  /graph/insights/{zone}           <- zone GraphRAG
GET  /grid-image/{zone}/{index}       <- serve panel image
```